# Movie review sentiment

Can a simple model tell a positive review from a negative one? This notebook trains a logistic regression on a tiny, hand-written set of reviews.

## 1. Load the reviews

In [1]:
import pandas as pd

reviews = [
    ("A great movie, I loved it", 1),
    ("Great acting and a fun story", 1),
    ("I loved the music, a great film", 1),
    ("Fun, warm and great to watch", 1),
    ("A good story and great acting", 1),
    ("Not bad at all, I loved it", 1),
    ("The best film this year, really good", 1),
    ("Good music, fun plot, I loved it", 1),
    ("A really fun movie with good acting", 1),
    ("Great film, the ending was good", 1),
    ("I loved the story, really great", 1),
    ("Good fun from start to finish", 1),
    ("A boring movie, I hated it", 0),
    ("Bad acting and a boring story", 0),
    ("I hated the music, an awful film", 0),
    ("Boring, slow and bad to watch", 0),
    ("A weak story and bad acting", 0),
    ("Not good at all, I hated it", 0),
    ("The worst film this year, really bad", 0),
    ("Bad music, boring plot, I hated it", 0),
    ("A really boring movie with bad acting", 0),
    ("Awful film, the ending was bad", 0),
    ("Not funny, the story was boring", 0),
    ("Not good, a waste of time", 0),
]
df = pd.DataFrame(reviews, columns=["text", "label"])
print(df.shape)
df.head()

(24, 2)


,text,label
0,"A great movie, I loved it",1
1,Great acting and a fun story,1
2,"I loved the music, a great film",1
3,"Fun, warm and great to watch",1
4,A good story and great acting,1


In [2]:
df["label"].value_counts()

label
1    12
0    12
Name: count, dtype: int64

## 2. Clean the text

Lowercase, drop punctuation and stop words, but **keep negations**: without *not*, "not good" would look like "good".

In [3]:
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

NEGATIONS = {"not", "no", "never", "nor"}
STOP_WORDS = ENGLISH_STOP_WORDS - NEGATIONS

def clean(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    words = [w for w in text.split() if w not in STOP_WORDS]
    return " ".join(words)

df["clean"] = df["text"].apply(clean)
example = df.loc[17]
lower = example["text"].lower()
letters = re.sub(r"[^a-z\s]", " ", lower)
print(example["text"])
print(lower)
print(letters)
print(example["clean"])

Not good at all, I hated it
not good at all, i hated it
not good at all  i hated it
not good hated


## 3. Split into train and test

A stratified split keeps the same share of positive and negative reviews in both parts.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["clean"], df["label"], test_size=0.25, stratify=df["label"], random_state=42
)
print(len(X_train), len(X_test))
print("train:", y_train.value_counts().to_dict())
print("test:", y_test.value_counts().to_dict())

18 6
train: {0: 9, 1: 9}
test: {0: 3, 1: 3}


## 4. Turn words into numbers

Two vectorizers: `CountVectorizer` (bag of words: how many times each word appears) and `TfidfVectorizer` (counts weighted down for words that appear everywhere). Unigrams and bigrams, so *not good* becomes a feature of its own.

In [5]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

bow = CountVectorizer(ngram_range=(1, 2))
tfidf = TfidfVectorizer(ngram_range=(1, 2))
Xtr_bow = bow.fit_transform(X_train)
Xte_bow = bow.transform(X_test)
Xtr_tfidf = tfidf.fit_transform(X_train)
Xte_tfidf = tfidf.transform(X_test)
print("BoW:", Xtr_bow.shape, "TF-IDF:", Xtr_tfidf.shape)

BoW: (18, 70) TF-IDF: (18, 70)


In [6]:
sample = X_train.iloc[:3].tolist()
words = ["good", "great", "bad", "boring", "not"]
small = CountVectorizer(vocabulary=words)
print(sample)
pd.DataFrame(small.transform(sample).toarray(), columns=words, index=["doc 1", "doc 2", "doc 3"])

['awful film ending bad', 'good story great acting', 'great acting fun story']


,good,great,bad,boring,not
doc 1,0,0,1,0,0
doc 2,1,1,0,0,0
doc 3,0,1,0,0,0


## 5. Train a logistic regression

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

results = {}
for name, (Xtr, Xte) in {"BoW": (Xtr_bow, Xte_bow), "TF-IDF": (Xtr_tfidf, Xte_tfidf)}.items():
    model = LogisticRegression(max_iter=1000)
    model.fit(Xtr, y_train)
    results[name] = (accuracy_score(y_train, model.predict(Xtr)), accuracy_score(y_test, model.predict(Xte)))
for name, (train_acc, test_acc) in results.items():
    print(f"{name}: train {train_acc:.3f}  test {test_acc:.3f}")

BoW: train 1.000  test 0.833
TF-IDF: train 1.000  test 0.833


## 6. Evaluate on the test set

In [8]:
from sklearn.metrics import classification_report, confusion_matrix

model = LogisticRegression(max_iter=1000).fit(Xtr_bow, y_train)
y_pred = model.predict(Xte_bow)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["negative", "positive"], digits=3))
for text, true, pred in zip(X_test, y_test, y_pred):
    if true != pred:
        print("Wrong:", repr(text), "is", true, "but the model said", pred)

[[2 1]
 [0 3]]
              precision    recall  f1-score   support

    negative      1.000     0.667     0.800         3
    positive      0.750     1.000     0.857         3

    accuracy                          0.833         6
   macro avg      0.875     0.833     0.829         6
weighted avg      0.875     0.833     0.829         6

Wrong: 'worst film year really bad' is 0 but the model said 1


## 7. Which words decide?

Each feature gets one coefficient: positive pushes towards *positive*, negative towards *negative*.

In [9]:
coefs = pd.Series(model.coef_[0], index=bow.get_feature_names_out()).sort_values()
print(coefs.tail(4).round(3))
print(coefs.head(4).round(3))
print(coefs[["film", "year", "really"]].round(3))

not bad    0.345
fun        0.439
loved      0.611
great      0.892
dtype: float64
boring       -0.607
hated        -0.530
bad acting   -0.470
not good     -0.432
dtype: float64
film      0.011
year      0.175
really    0.310
dtype: float64


## 8. How the model scores one review

The score is the intercept plus, for every word of the review, its count times its weight. The sigmoid turns the score into a probability of *positive*.

In [10]:
import numpy as np

text = "worst film year really bad"
x = bow.transform([text]).toarray()[0]
names = bow.get_feature_names_out()
print(f"intercept {model.intercept_[0]:+.3f}")
for i in np.flatnonzero(x):
    print(f"{names[i]:<12} count {x[i]}  weight {model.coef_[0][i]:+.3f}")
print("worst in vocabulary:", "worst" in bow.vocabulary_)
score = model.decision_function(bow.transform([text]))[0]
print(f"score {score:.3f}  probability {model.predict_proba(bow.transform([text]))[0][1]:.3f}")

intercept -0.075
bad          count 1  weight -0.332
film         count 1  weight +0.011
film year    count 1  weight +0.175
really       count 1  weight +0.310
year         count 1  weight +0.175
year really  count 1  weight +0.175
worst in vocabulary: False
score 0.438  probability 0.608


## 9. Moving the threshold

The model says *positive* when the probability is at least 0.5. Each test review with its probability:

In [11]:
proba = model.predict_proba(Xte_bow)[:, 1]
for text, true, p in sorted(zip(X_test, y_test, proba), key=lambda row: row[2]):
    print(f"{p:.3f}  {true}  {text}")

0.156  0  bad music boring plot hated
0.292  0  boring slow bad watch
0.608  0  worst film year really bad
0.652  1  good fun start finish
0.748  1  good music fun plot loved
0.815  1  loved music great film
